# Parsing one page: Docling, PaddleOCR-VL and Qwen VL

Notebook demo of the AI2Future workshop *Can We Rely on Extracted Documents? Parsing Methods Compared*
(Filip Pavičić, Atomic Intelligence).

- **What:** one page of a synthetic lab report, followed through a parsing pipeline.
- **This copy runs every model live:** Docling, PaddleOCR-VL and Qwen VL, general and fine-tuned, each
  printing how long it took. It needs a GPU runtime; the demo itself is `demo.ipynb`.
- **The fine-tuned weights are not public:** set `DUDEMO_FINETUNED` to their folder or Hugging Face repo id;
  without it, that model is loaded from `data/cached/`.

<details><summary>What is where</summary>

Loading files and drawing are in `src/dudemo/`. Everything the demo is about is in the cells below.
</details>

## Setup

- **Colab:** choose a GPU runtime first (Runtime → Change runtime type). The first cell clones the
  repository and installs the dependencies and PaddleOCR-VL.
- **Locally:** install as described in the README under "Every model live", then start Jupyter from the
  repository.
- **The first run of each model downloads its weights:** about 7 GB for each Qwen VL model.

In [ ]:
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and Path.cwd().name != "document-understanding-demo":
    if not Path("document-understanding-demo").exists():
        !git clone --quiet --depth 1 https://github.com/Atomic-Intelligence/document-understanding-demo.git
    %cd document-understanding-demo
    # pip warns that google-colab pins an older requests than Docling needs; Colab works with the
    # newer one, so the output is shown only if the install fails.
    pip = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--progress-bar", "off", "-e", "."],
                         capture_output=True, text=True)
    if pip.returncode != 0:
        print(pip.stdout, pip.stderr)
        raise RuntimeError("Installing the dependencies failed; pip's output is above.")
    print("Dependencies installed.")
    import torch
    if torch.cuda.is_available() and torch.version.cuda.startswith("12."):   # PaddleOCR-VL runs too
        # paddlepaddle-gpu pins exact CUDA 12.6 libraries, which would replace the newer ones Colab's torch
        # uses. It runs on those, so it is installed without its dependencies.
        for args in (["paddlepaddle-gpu==3.2.0", "--no-deps", "-i", "https://www.paddlepaddle.org.cn/packages/stable/cu126/"],
                     ["httpx", "networkx", "opt-einsum", "protobuf", "-e", ".[live]"]):
            pip = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--progress-bar", "off", *args],
                                 capture_output=True, text=True)
            if pip.returncode != 0:
                print(pip.stdout, pip.stderr)
                raise RuntimeError("Installing PaddleOCR-VL failed; pip's output is above.")
        print("PaddleOCR-VL installed.")
    else:
        print("No GPU with CUDA 12 here: PaddleOCR-VL is loaded from data/cached/.")

# The repository root, whether Jupyter was started in notebooks/ or in the root
ROOT = Path.cwd() if (Path.cwd() / "src" / "dudemo").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import difflib
import json
import re
import time

import pandas as pd
from PIL import ImageDraw

from dudemo import data, quiet, show

quiet()   # no model-loading logs in the output

In [ ]:
from dudemo import live

data.LIVE = set(live.available())   # every model that can run here; any other is loaded from data/cached/
print("Live:", ", ".join(data.NAMES[model] for model in data.CACHED_MODELS if model in data.LIVE) or "nothing")

## 1. The page

### 1.1 · The report

- A synthetic Croatian blood analysis report.
- Patient fields, two tables of results, units, reference ranges, and a flag column (H high, L low).
- An image only, with no text layer, so every parser has to read the pixels.

**Question for the room:** what would a parser get wrong on this page?

In [ ]:
PAGE = "lab_report_1"
page = data.page_image(PAGE)
show.image(page, f"{PAGE}.png")

### 1.2 · The ground truth

A JSON file next to the page. Each region has:

- `type`: heading, paragraph, key-value section or table;
- `box`: `[x0, y0, x1, y1]`, in pixels from the top-left corner;
- its content: `text`, key-value `pairs`, or a table as `html`.

One region of each type:

In [ ]:
ground_truth = json.loads((data.DATA / "ground_truth" / f"{PAGE}.json").read_text(encoding="utf-8"))

seen = set()
for region in ground_truth["regions"]:
    if region["type"] not in seen:
        seen.add(region["type"])
        print(json.dumps(region, ensure_ascii=False)[:260], "\n")

### 1.3 · One record, many views

- The JSON file is the lossless record of the page.
- Every other format is generated from it, for one use:
  - an image with boxes, to check the layout;
  - Markdown, to read and see the whole page;
  - text chunks with their page and box, for a search index or a RAG system.

**Image with boxes:** one rectangle per region, in its type's colour.

In [ ]:
COLORS = {"heading": "#E07B39", "paragraph": "#7F95A6", "key_value_section": "#5E9E1F", "table": "#058ACC"}

drawn = page.copy()
pen = ImageDraw.Draw(drawn)
for region in ground_truth["regions"]:
    pen.rectangle(region["box"], outline=COLORS[region["type"]], width=4)
show.image(drawn, "The ground truth, drawn: heading orange, text grey, key-value green, table blue")

### 1.4 · Markdown

The regions in reading order, each in the usual Markdown form for its type. Rendered, it is the target
every parser below is held against.

In [ ]:
def table_to_markdown(rows):
    header, *body = rows
    lines = ["| " + " | ".join(header) + " |", "|" + " --- |" * len(header)]
    lines += ["| " + " | ".join(row) + " |" for row in body]
    return "\n".join(lines)


def to_markdown(regions):
    parts = []
    for region in regions:                      # the file's order is the reading order
        if region["type"] == "heading":
            parts.append("## " + region["text"])
        elif region["type"] == "key_value_section":
            parts.append("\n".join(f"**{key}** {value}  " for key, value in region["pairs"]))
        elif region["type"] == "table":
            parts.append(table_to_markdown(data.html_rows(region["html"])))
        else:
            parts.append(region["text"])
    return "\n\n".join(parts)


show.markdown(to_markdown(ground_truth["regions"]), "Target Markdown")

### 1.5 · Zoom

Check a value against the pixels: a row of the page, enlarged, under what the file holds for it.

- Watch these two rows: they come back in section 5.

In [ ]:
truth = data.ground_truth(PAGE)
for label in ["Leukociti (WBC)", "ALT (ALAT)"]:
    x0, y0, x1, y1 = data.row_box(PAGE, label)
    row = page.crop((x0, y0 - 2, x1, y1 + 3))
    _header, cells = show.find_row(truth.tables, label)
    show.image(row.resize((row.width * 2, row.height * 2)), "The file says:  " + "  |  ".join(cells))

## 2. Parse it with Docling

### 2.1 · Settings

The settings of the evaluation, each commented with its step. The first run downloads the models.

In [ ]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (LayoutObjectDetectionOptions, PdfPipelineOptions,
                                                RapidOcrOptions, TableFormerMode, TableStructureOptions)
from docling.document_converter import DocumentConverter, ImageFormatOption, PdfFormatOption

options = PdfPipelineOptions(
    layout_options=LayoutObjectDetectionOptions.from_preset("layout_heron_default"),  # 1. detection
    do_ocr=True,
    ocr_options=RapidOcrOptions(backend="torch"),                                     # 2. recognition: text
    do_table_structure=True,
    table_structure_options=TableStructureOptions(mode=TableFormerMode.ACCURATE),     # 2. recognition: tables
    generate_page_images=True,                                                       # to draw the regions
)
converter = DocumentConverter(format_options={
    InputFormat.IMAGE: ImageFormatOption(pipeline_options=options),
    InputFormat.PDF: PdfFormatOption(pipeline_options=options),
})
converter.initialize_pipeline(InputFormat.IMAGE)   # loads the models
print("Docling is ready.")

### 2.2 · Run

Docling reads the page, on this machine's CPU.

In [ ]:
start = time.perf_counter()
doc = converter.convert(data.page_path(PAGE)).document
print(f"Docling read the page in {time.perf_counter() - start:.0f} seconds on this machine.")

### 2.3 · Detection

The regions Docling found, in the same colours as the ground truth.

- Each patient label and each value is its own text region.

In [ ]:
docling = data.from_docling(doc)
show.regions(docling, "Docling: regions by type")

### 2.4 · Recognition

The second table as it is on the page, then the rows and columns TableFormer made of it.

In [ ]:
show.image(page.crop(data.table_box(PAGE, "Glukoza (na tašte)", pad=6)), "On the page")
show.table(doc.tables[1].export_to_dataframe(doc=doc))

### 2.5 · Transformation

- Docling keeps its result as a DoclingDocument, its lossless record, like the JSON file in 1.2.
- The regions in 2.3 and this Markdown are two views of it.

In [ ]:
show.markdown(docling, "Docling's Markdown")

Reading the Markdown shows at once where Docling struggles:

- the patient fields come out as loose lines, not as pairs;
- the tables get an extra column, with each range split over two cells;
- 10⁹/L becomes 109/L.

## 3. The same page, two other parsers

- Their outputs were made on a GPU and are loaded from `data/cached/`.
- Each one: its regions on the page, next to its Markdown, rendered.

### 3.1 · PaddleOCR-VL

- It finds the header and the footer (teal). Its own Markdown export leaves them out; the Markdown
  here is written from all its blocks, so they are in.
- It has no key-value class, so it writes the patient fields as a table; each label stays next to its
  value.
- Its tables have the right columns; the units come back as LaTeX.

In [ ]:
paddle = data.cached(PAGE, "paddleocr_vl")
show.parse(paddle)

### 3.2 · Qwen VL, general

- It writes each block as a tag with its type, its box and its text; 4.1 shows the tags.
- This is the model as released: its boxes are not read from the page.

<details><summary>The prompt, in full</summary>

The same prompt goes with every page, for this model and for the fine-tuned one in section 4.

```text
### Doctags Format Specification 

* Supported block types:

  1. `<heading>...</heading>`
  2. `<paragraph>...</paragraph>`
  3. `<table>...</table>`
  4. `<key_value_section>...</key_value_section>`
  5. `<list>...</list>`
  6. `<omr_question>...</omr_question>`

* For every block:

  * Always include a `<location>` tag immediately after the opening tag with attributes:

    * `<x1>, <y1>, <x2>, <y2>` → bounding box coordinates of the block (top-left and bottom-right corners).
  * Preserve reading order: top-to-bottom, left-to-right.
  * Do not merge unrelated blocks. Each logical region should be a separate doctag.

* Content rules:

  * Text must match the document exactly (numbers, symbols, punctuation, capitalization).
  * Do not summarize, paraphrase, or translate. Keep the original text.
  * Retain line breaks inside a block only if they are semantically relevant (e.g., table rows, list items).

* Block-specific rules:

  * `<heading>`: Section titles, headers, or document titles.
  * `<paragraph>`: Continuous prose text.
  * `<table>`: Transcribe tables using html.
  * `<key_value_section>`: For fields with labels and values. Format each as `Key: Value`, one per line.
  * `<list>`: For bullet or numbered lists. Prefix each item with `-` or the number.
  * `<omr_question>`: Return the question, answer options if there are any, and the selected answer.

* Output Restrictions (Important):

  * Do NOT output Markdown formatting, code blocks, XML headers, comments, or explanations.
  * Do NOT add metadata or instructions outside the doctags.
  * The output must only be the doctags representation.


### Example 

<heading>  
<location><x1>60.0</x1><y1>80.0</y1><x2>560.0</x2><y2>120.0</y2></location>  
<text>
Annual Business Report 2023
</text>
</heading>  

<paragraph>  
<location><x1>65.0</x1><y1>140.0</y1><x2>545.0</x2><y2>240.0</y2></location>  
<text>
The fiscal year 2023 was marked by strong growth and improved profitability. Revenue increased by 8.5% to $15,250,000, driven by successful product launches and expansion into new markets.
</text>
</paragraph>  

<list>  
<location><x1>70.0</x1><y1>270.0</y1><x2>470.0</x2><y2>360.0</y2></location>  
<text>
- Revenue growth of 8.5%
- Gross margin improved to 45.2%  
- Operating income up by 12%
</text>
</list>  

<table>  
<location><x1>75.0</x1><y1>400.0</y1><x2>575.0</x2><y2>520.0</y2></location>  
<text>
<table><tr><th>Year</th><th>Revenue</th><th>Operating Income</th></tr><tr><td>2022</td><td>$14,050,000</td><td>$1,875,000</td></tr><tr><td>2023</td><td>$15,250,000</td><td>$2,100,000</td></tr></table>
</text>
</table>  

<key_value_section>
<location><x1>80.0</x1><y1>550.0</y1><x2>430.0</x2><y2>650.0</y2></location>
<text>
Client: ACME Corporation
Invoice Number: 2024-05-01
Due Date: 07/30/2024
</text>
</key_value_section>

<omr_question>
<location><x>62.0</x><y>595.0</y><w>547.0</w><h>81.0</h></location>
<text>
Question: Koliko ste zadovoljni kvalitetom naše usluge?
Selected answer: Zadovoljan
Answer options: 
<option>Vrlo nezadovoljan</option><option>Nezadovoljan</option><option>Neutralan</option><option>Zadovoljan</option><option>Vrlo zadovoljan</option>
</text>
</omr_question>
```
</details>

In [ ]:
qwen_general = data.cached(PAGE, "qwen_base")
show.parse(qwen_general)

### 3.3 · Three parsers, one page

**Question for the room:** which one would you start with for documents like this one?

In [ ]:
show.side_by_side(docling, paddle, qwen_general)

## 4. Fine-tuning Qwen VL

- The general model from 3.2 against the same model after our fine-tuning.
- The same prompt and settings; only the weights differ.

### 4.1 · The first tag and its box

- The page title, as each model wrote it. `<location>` is its box: `x1`, `y1` the top-left corner,
  `x2`, `y2` the bottom-right, in pixels. Below, that box on the page.
- General: a made-up box, and closing tags that don't match (red). Fine-tuned: a box on the title.

In [ ]:
qwen_finetuned = data.cached(PAGE, "qwen_finetuned")
show.first_tag(PAGE, qwen_general, qwen_finetuned)

### 4.2 · Regions

- General: every box starts near the top-left corner of the page.
- Fine-tuned: a box on each region, with its type; the report title and the section titles are
  headings.

In [ ]:
show.side_by_side(qwen_general, qwen_finetuned)

### 4.3 · What fine-tuning fixed

- Every value on this page that one of the models gets wrong; the fine-tuned model has each of them
  right.
- Two are flags: a normal AST marked high (H), and the high cholesterol left unmarked.

In [ ]:
show.wrong_values(PAGE, qwen_general, qwen_finetuned)

### 4.4 · What fine-tuning did not fix

- The doctor's field on a second report. Each model writes a different name, and neither is the one on
  the page.
- Nothing in either output marks the name as wrong: a silent error (section 5).

In [ ]:
PAGE_2 = "lab_report_2"
show.field_error(PAGE_2, "LIJEČNIK:", data.cached(PAGE_2, "qwen_base"), data.cached(PAGE_2, "qwen_finetuned"))

## 5. Three kinds of error

- One row of the page for each kind: the row enlarged, as the page has it, and as a parser wrote it.
- Red: a cell that is not on the page.

### 5.1 · Visibly broken

- Anyone reading the output sees it: PaddleOCR-VL writes the unit as LaTeX.

In [ ]:
show.row_error(PAGE, "Leukociti (WBC)", paddle)

### 5.2 · Structure wrong

- The text is right, the shape is not: Docling splits the range over two columns, so the table has a
  column the page does not have.

In [ ]:
show.row_error(PAGE, "Glukoza (na tašte)", docling)

### 5.3 · Silent

- Well-formed, and looks right: Docling reverses the sign of the limit, so below the limit becomes
  above it. It also moves the limit into the extra column from 5.2.
- Nothing in the output marks it, like the doctor's name in 4.4.

In [ ]:
show.row_error(PAGE, "ALT (ALAT)", docling)

## 6. Checks after extraction

### 6.1 · Rules

Three rules for lab results:

- the unit is a known unit;
- the range looks like a range;
- the H or L flag fits the result and the range.

In [ ]:
UNITS = {"10⁹/L", "10¹²/L", "g/L", "mmol/L", "µmol/L", "μmol/L", "U/L", "%", "fL", "pg", "L/L", "mg/L"}
NUMBER = r"\d+(?:,\d+)?"
RANGE = re.compile(rf"(?P<low>{NUMBER}) - (?P<high>{NUMBER})|(?P<sign>[<>]) (?P<limit>{NUMBER})")


def number(text):
    return float(text.replace(",", "."))


def unit_is_known(unit):
    return unit in UNITS


def range_looks_like_a_range(reference):
    return RANGE.fullmatch(reference) is not None


def flag_fits(result, reference, flag):
    """H means above the range, L below it. A row without a flag is not checked."""
    match = RANGE.fullmatch(reference)
    if flag not in ("H", "L") or not match or not re.fullmatch(NUMBER, result):
        return None
    value = number(result)
    if match["low"]:
        expected = "L" if value < number(match["low"]) else "H" if value > number(match["high"]) else ""
    elif match["sign"] == "<":
        expected = "H" if value > number(match["limit"]) else ""
    else:
        expected = "L" if value < number(match["limit"]) else ""
    return expected == flag

### 6.2 · Docling's tables through the rules

- Only the rows a rule sends to a person, with the rule that fails; every other row passes.
- The reversed sign in the ALT row (5.3) is caught only because the page also prints an H, which no
  longer fits the limit.

In [ ]:
rows = []
for table in docling.tables:
    header = table[0]
    result_column, unit_column = header.index("Rezultat"), header.index("Jedinica")
    for row in table[1:]:
        result, unit, flag = row[result_column], row[unit_column], row[-1]
        reference = " ".join(cell for cell in row[unit_column + 1:-1] if cell)   # a split range as written
        failed = []
        if not unit_is_known(unit):
            failed.append("unit is not known")
        if not range_looks_like_a_range(reference):
            failed.append("range is not a range")
        if flag_fits(result, reference, flag) is False:
            failed.append("flag does not fit")
        if failed:
            rows.append({"Parameter": row[0], "Result": result, "Unit": unit, "Range": reference, "Flag": flag,
                         "Sent to a person because": "✗ " + "; ".join(failed)})
show.table(pd.DataFrame(rows))

## 7. Two parsers, rules and a person

- The second report from 4.4. On the first, fine-tuned Qwen VL gets every field and table value right;
  on this one, both parsers make silent errors.
- Two parsers read every field, table value, title and paragraph; the rules of 6.1 and a person decide
  where they disagree; then the score against the ground truth.

### 7.1 · Where the two parsers disagree

- Fine-tuned Qwen VL and PaddleOCR-VL agree on almost all of the page. These are the places where they
  do not; in a paragraph, only the words that differ are shown.

In [ ]:
qwen_2, paddle_2 = data.cached(PAGE_2, "qwen_finetuned"), data.cached(PAGE_2, "paddleocr_vl")
readings = data.readings(PAGE_2, qwen_2, paddle_2, text=True)   # labels, headers and texts by the closest match

show.disagreements(readings, ["Qwen VL, fine-tuned", "PaddleOCR-VL"])

### 7.2 · Rules and domain knowledge decide

- Where the two disagree, a reading that breaks a rule loses: an unknown unit, a range that is not a
  range, or a word Spol cannot say.
- What no rule decides is left for a person: a name, or the wording of a paragraph.

In [ ]:
SEX = {"Ž", "M", "ŽENSKO", "MUŠKO", "ŽENSKI", "MUŠKI", "ŽENA", "MUŠKARAC"}   # what Spol can say


def passes(where, value):
    """A reading passes when it is there and fits what the page can say at that place."""
    if value is None:
        return False
    if where.upper().startswith("SPOL"):
        return value.upper() in SEX
    if where.endswith("Jedinica"):
        return unit_is_known(value)
    if where.endswith("Referentni interval"):
        return range_looks_like_a_range(value)
    return True


decided = []
for where, (qwen, paddle) in readings:
    if qwen is not None and data.same(qwen, paddle):
        decided.append((where, qwen, "same"))
        continue
    valid = [value for value in (qwen, paddle) if passes(where, value)]
    decided.append((where, valid[0], "rule") if len(valid) == 1 else (where, None, "person"))

show.decisions(PAGE_2, decided)

### 7.3 · Left for a person

- No rule knows a name, or which of two words is right: the page is the only answer.
- In a paragraph, the person reads only the words where the two parsers differ, in bold.

In [ ]:
show.for_a_person(PAGE_2, decided, readings, ["Qwen VL, fine-tuned", "PaddleOCR-VL"])

### 7.4 · The score

- The person writes what the page says: a value, or the words of a paragraph where the parsers differ,
  in order. Then every place of the page is held against the ground truth: each field, table value,
  title and paragraph.

In [ ]:
person = {                                   # what the person reads on the page above
    "LIJEČNIK:": "Dr. Ivić",
    "Text: NAPOMENA: * Povišene vrijednosti …": ["prilagodbu."],
    "Text: Pronađene su povišene koncentracije …": ["Pronađene", "daljnjeg", "praćenja"],
    "Text: (Ivan Perić)": ["Perić)"],
    "Text: Ovaj nalaz je izrađen …": ["nalaza", "ovlaštene", "intervali"],
}

final = data.with_answers(decided, readings, person)
show.score(PAGE_2, final)

## 8. Try your own document

- Set `MY_FILE` to the path of a PDF or an image, and run the cell.
- On Colab, upload the file with the file browser on the left; its path is `/content/<file name>`.

Links: [Docling](https://github.com/docling-project/docling) ·
[PaddleOCR-VL](https://huggingface.co/PaddlePaddle/PaddleOCR-VL-1.6) ·
[Qwen2.5-VL](https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct)

In [ ]:
MY_FILE = "path/to/your/document.pdf"

if Path(MY_FILE).exists():
    show.parse(data.from_docling(converter.convert(MY_FILE).document))
else:
    print("Set MY_FILE to the path of a PDF or an image, then run this cell again.")